In [1]:
import stim
import itertools
import numpy as np
import pickle
import time
import os
import re
import pprint
import numpy as np

from autqec.automorphisms   import *
from autqec.utils.qec       import *
from autqec.utils.qiskit    import *
from autqec.graph_auts      import *
from autqec.ZX_dualities    import *
from autqec.ZY_dualities    import *
from autqec.magma_interface import *
from autqec.code_embedding  import *

from magma_online           import *
from stim_helpers           import *
from compare_circuits       import *
from cws_helpers            import * 


### 5-qubit code

In [2]:
generators_5qubits = [
    stim.PauliString("XZZXI"),
    stim.PauliString("IXZZX"),
    stim.PauliString("XIXZZ"),
    stim.PauliString("ZZXIX")
]

# logical operators
logical_I = stim.PauliString("IIIII")
logical_X = stim.PauliString("XXXXX")
logical_Z = stim.PauliString("ZZZZZ")
logical_Y = canonicalize_real_pauli(logical_X * logical_Z)

logical_X_min = stim.PauliString("-YIXIY")
logical_Z_min = stim.PauliString("-XIZIX")
logical_Y_min = canonicalize_real_pauli(logical_X_min * logical_Z_min)

logical_basis5 = {
    "I": logical_I,
    "X": canonicalize_real_pauli(logical_X_min),
    "Y": canonicalize_real_pauli(logical_Y_min),
    "Z": canonicalize_real_pauli(logical_Z_min)
}

### [[15,3,3]] stabilizer generators: 3 independent copies

In [3]:
generators_5qubits_3blocks = []

for s in generators_5qubits:
    generators_5qubits_3blocks.append(s + logical_I + logical_I)
    generators_5qubits_3blocks.append(logical_I + s + logical_I)
    generators_5qubits_3blocks.append(logical_I + logical_I + s)

# single-block logicals on each encoded qubit
logical_XII = logical_X_min + logical_I + logical_I
logical_ZII = logical_Z_min + logical_I + logical_I

logical_IXI = logical_I + logical_X_min + logical_I
logical_IZI = logical_I + logical_Z_min + logical_I

logical_IIX = logical_I + logical_I + logical_X_min
logical_IIZ = logical_I + logical_I + logical_Z_min

logical_YII = canonicalize_real_pauli(logical_XII * logical_ZII)
logical_IYI = canonicalize_real_pauli(logical_IXI * logical_IZI)
logical_IIY = canonicalize_real_pauli(logical_IIX * logical_IIZ)

# convenient logical basis for generators
logical_basis_3_generators = {
    "XII": canonicalize_real_pauli(logical_XII),
    "YII": canonicalize_real_pauli(logical_YII),
    "ZII": canonicalize_real_pauli(logical_ZII),

    "IXI": canonicalize_real_pauli(logical_IXI),
    "IYI": canonicalize_real_pauli(logical_IYI),
    "IZI": canonicalize_real_pauli(logical_IZI),

    "IIX": canonicalize_real_pauli(logical_IIX),
    "IIY": canonicalize_real_pauli(logical_IIY),
    "IIZ": canonicalize_real_pauli(logical_IIZ),
}

# full 3-qubit Pauli logical basis, excluding III
logical_basis_3blocks = {}

for a in ["I", "X", "Y", "Z"]:
    for b in ["I", "X", "Y", "Z"]:
        for c in ["I", "X", "Y", "Z"]:
            label = a + b + c
            if label == "III":
                continue

            pauli = (
                  (logical_basis5[a] if a != "I" else logical_I)
                + (logical_basis5[b] if b != "I" else logical_I)
                + (logical_basis5[c] if c != "I" else logical_I)
            )
            logical_basis_3blocks[label] = canonicalize_real_pauli(pauli)

print_combined_stabilizers(
    generators_5qubits_3blocks,
    "Combined generators: three blocks of [[5,1,3]] -> [[15,3,3]]",
    num_blocks = 3
)

print("Number of stabilizer generators: ",  len(generators_5qubits_3blocks))
print("Number of logical basis elements:", len(logical_basis_3blocks))


Combined generators: three blocks of [[5,1,3]] -> [[15,3,3]]

+XZZXIIIIIIIIIII   +IIIIIXZZXIIIIII   +IIIIIIIIIIXZZXI
+IXZZXIIIIIIIIII   +IIIIIIXZZXIIIII   +IIIIIIIIIIIXZZX
+XIXZZIIIIIIIIII   +IIIIIXIXZZIIIII   +IIIIIIIIIIXIXZZ
+ZZXIXIIIIIIIIII   +IIIIIZZXIXIIIII   +IIIIIIIIIIZZXIX

Number of stabilizer generators:  12
Number of logical basis elements: 63


In [4]:
def get_prologue_circuit(block = 0, qubits = 5):
    prologue = stim.Circuit()
    prologue.append("H", [0 + qubits * block])
    prologue.append("S", [0 + qubits * block])
    prologue.append("Y", [2 + qubits * block])
    prologue.append("H", [4 + qubits * block])
    prologue.append("S", [4 + qubits * block])
    return prologue


def get_epilogue_circuit(block = 0, qubits = 5):
    epilogue = stim.Circuit()
    epilogue.append("H", [0 + qubits * block])
    epilogue.append("S_DAG", [0 + qubits * block])
    epilogue.append("Y", [2 + qubits * block])
    epilogue.append("H", [4 + qubits * block])
    epilogue.append("S_DAG", [4 + qubits * block])
    return epilogue


prologue_3blocks = get_prologue_circuit(0) + get_prologue_circuit(1) + get_prologue_circuit(2)
epilogue_3blocks = get_epilogue_circuit(0) + get_epilogue_circuit(1) + get_epilogue_circuit(2)

# round robin: for i,j,k in {0,2,4} in each block, CZZ(i,j,k)
RR_circuit_a = []
RR_circuit_a.append(("CCZ", [0,5,10]))
RR_circuit_a.append(("CCZ", [0,5,12]))
RR_circuit_a.append(("CCZ", [0,7,10]))
RR_circuit_a.append(("CCZ", [0,7,12]))
RR_circuit_a.append(("CCZ", [2,5,10]))
RR_circuit_a.append(("CCZ", [2,5,12]))
RR_circuit_a.append(("CCZ", [2,7,10]))
RR_circuit_a.append(("CCZ", [2,7,12]))
RR_circuit_a.append(("CCZ", [4,9,14]))

RR_circuit_b = []
RR_circuit_b.append(("CCZ", [0,5,14]))
RR_circuit_b.append(("CCZ", [0,7,14]))
RR_circuit_b.append(("CCZ", [2,5,14]))
RR_circuit_b.append(("CCZ", [2,7,14]))
RR_circuit_b.append(("CCZ", [4,9,10]))
RR_circuit_b.append(("CCZ", [4,9,12]))

RR_circuit_c = []
RR_circuit_c.append(("CCZ", [0,9,10]))
RR_circuit_c.append(("CCZ", [0,9,12]))
RR_circuit_c.append(("CCZ", [2,9,10]))
RR_circuit_c.append(("CCZ", [2,9,12]))
RR_circuit_c.append(("CCZ", [4,5,14]))
RR_circuit_c.append(("CCZ", [4,7,14]))

RR_circuit_d = []
RR_circuit_d.append(("CCZ", [4,5,10]))
RR_circuit_d.append(("CCZ", [4,5,12]))
RR_circuit_d.append(("CCZ", [4,7,10]))
RR_circuit_d.append(("CCZ", [4,7,12]))
RR_circuit_d.append(("CCZ", [0,9,14]))
RR_circuit_d.append(("CCZ", [2,9,14]))

In [5]:
def update_logical_basis_by_circuit(logical_basis: dict, circuit: stim.Circuit) -> dict:
    """
    Conjugate every logical operator in a basis dict by a Stim circuit.
    Returns a new dict with the same keys and updated PauliStrings.
    """
    return {
        label: canonicalize_real_pauli(conjugate_stabilizer_by_circuit(op, circuit))
        for label, op in logical_basis.items()
    }

In [6]:
# code after prologue

generators_5qubits_3blocks_after_prologue = []

for s in generators_5qubits_3blocks:
    generators_5qubits_3blocks_after_prologue.append(conjugate_stabilizer_by_circuit(s, prologue_3blocks))

print_combined_stabilizers(
    generators_5qubits_3blocks_after_prologue,
    "Combined generators after prologue circuit: three blocks of [[5,1,3]] -> [[15,3,3]]",
    num_blocks = 3
)

logical_basis_3blocks_after_prologue = update_logical_basis_by_circuit(logical_basis_3blocks, prologue_3blocks)


Combined generators after prologue circuit: three blocks of [[5,1,3]] -> [[15,3,3]]

-ZZZXIIIIIIIIIII   -IIIIIZZZXIIIIII   -IIIIIIIIIIZZZXI
-IXZZZIIIIIIIIII   -IIIIIIXZZZIIIII   -IIIIIIIIIIIXZZZ
-ZIXZYIIIIIIIIII   -IIIIIZIXZYIIIII   -IIIIIIIIIIZIXZY
-YZXIZIIIIIIIIII   -IIIIIYZXIZIIIII   -IIIIIIIIIIYZXIZ



### Converting to CWS Codes

In [29]:
graph_stabs, C, A, hadamard_qubits = stabilizer_code_to_cws(
    stabilizers = generators_5qubits_3blocks_after_prologue,
    logical_Zs  = [logical_basis_3blocks_after_prologue["IIZ"], logical_basis_3blocks_after_prologue["ZII"], logical_basis_3blocks_after_prologue["IZI"]],
    logical_Xs  = [logical_basis_3blocks_after_prologue["IIX"], logical_basis_3blocks_after_prologue["XII"], logical_basis_3blocks_after_prologue["IXI"]],
)

print("Graph stabilizers after prologue:")
print("Standard Form:")
for g in graph_stabs:
    print(str(g).replace("_", "I"))

print("Code Words =")
for c in C:
    print(c)

print("Adj Matrix =\n", A)


Graph stabilizers after prologue:
Standard Form:
+XZIZZIIIIIIIIII
+ZXIZIIIIIIIIIII
+IIXZZIIIIIIIIII
+ZZZXIIIIIIIIIII
+ZIZIXIIIIIIIIII
+IIIIIXZIZZIIIII
+IIIIIZXIZIIIIII
+IIIIIIIXZZIIIII
+IIIIIZZZXIIIIII
+IIIIIZIZIXIIIII
+IIIIIIIIIIXZIZZ
+IIIIIIIIIIZXIZI
+IIIIIIIIIIIIXZZ
+IIIIIIIIIIZZZXI
+IIIIIIIIIIZIZIX
Code Words =
000000000000000
000000000001001
010010000000000
010010000001001
000000100100000
000000100101001
010010100100000
010010100101001
Adj Matrix =
 [[0 1 0 1 1 0 0 0 0 0 0 0 0 0 0]
 [1 0 0 1 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 1 1 0 0 0 0 0 0 0 0 0 0]
 [1 1 1 0 0 0 0 0 0 0 0 0 0 0 0]
 [1 0 1 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 1 1 0 0 0 0 0]
 [0 0 0 0 0 1 0 0 1 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 1 0 0 0 0 0]
 [0 0 0 0 0 1 1 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 0 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 1 0 1 1]
 [0 0 0 0 0 0 0 0 0 0 1 0 0 1 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 1 1]
 [0 0 0 0 0 0 0 0 0 0 1 1 1 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 0 1 0 0]]


In [13]:
C_new, hyperedges = apply_ccz_to_cws(C, A, RR_circuit_a)

print("Updated codewords:")
for c, c_ in zip(C_new, C):
    if (c!=c_):
        print("not equal")
    print(c)

print("\nHypergraph:")
print_hypergraph(hyperedges)

Updated codewords:
000000000000000
000000000001001
010010000000000
010010000001001
000000100100000
000000100101001
010010100100000
010010100101001

Hypergraph:
  2-edges (18): [[0, 1], [0, 3], [0, 4], [1, 3], [2, 3], [2, 4], [5, 6], [5, 8], [5, 9], [6, 8], [7, 8], [7, 9], [10, 11], [10, 13], [10, 14], [11, 13], [12, 13], [12, 14]]
  3-edges (9): [[0, 5, 10], [0, 5, 12], [0, 7, 10], [0, 7, 12], [2, 5, 10], [2, 5, 12], [2, 7, 10], [2, 7, 12], [4, 9, 14]]


In [16]:
n = 15
k = 3
d = 3

command_file_name = f"./magma_commands_n{n}k{k}d{d}.txt"
output_file_name  = f"./magma_output_n{n}k{k}d{d}.txt"

# Use the three single-logical-qubit generator codewords
C_new_gens = [C_new[1], C_new[2], C_new[4]]

magma_lines = []
magma_lines.append("F := GF(2);")
magma_lines.append("G := Matrix(F, [")
magma_lines += [
    "  [" + ",".join(row) + "]" + ("," if i < len(C_new_gens) - 1 else "")
    for i, row in enumerate(C_new_gens)
]
magma_lines.append("]);")
magma_lines.append("C := LinearCode(G);")
magma_lines.append("A := AutomorphismGroup(C);")
magma_lines.append("A;")
magma_lines.append("Generators(A);")
magma_lines.append("#A;")

with open(command_file_name, "w") as f:
    f.write("\n".join(magma_lines))

run_magma_online_from_file(command_file_name)

with open(output_file_name, "r") as f:
    magma_output = f.read()

print(magma_output)


def parse_magma_permutation_group(output_text, n):
    """
    Parse Magma permutation generators into Python 0-indexed permutations of length n.
    Skips the set literal { ... } block that Magma prints for Generators(A).
    """
    # Remove the set literal { ... } before parsing
    cleaned = re.sub(r'\{[^}]*\}', '', output_text, flags=re.DOTALL)

    perms = []
    # Each generator is on its own line; split by newline first
    for line in cleaned.splitlines():
        line = line.strip()
        if not line or not line.startswith('('):
            continue

        perm = list(range(n))
        cycles = re.findall(r'\(([^)]+)\)', line)
        for cyc in cycles:
            cyc_nums = [int(x) - 1 for x in cyc.split(',')]
            for i in range(len(cyc_nums)):
                perm[cyc_nums[i]] = cyc_nums[(i + 1) % len(cyc_nums)]
        perms.append(perm)

    return perms


auts = parse_magma_permutation_group(magma_output, n=n)
print("num generators =", len(auts))
for p in auts:
    print(p)

Permutation group A acting on a set of cardinality 15
Order = 17418240 = 2^11 * 3^5 * 5 * 7
(7, 10)
(8, 9)
(11, 13)
(9, 11)
(7, 12)(10, 15)
(6, 8)
(13, 14)
(4, 6)
(3, 4)
(1, 3)
(2, 5)
(2, 10)(5, 7)
(12, 15)
{
(3, 4),
(2, 5),
(7, 10),
(6, 8),
(9, 11),
(13, 14),
(11, 13),
(4, 6),
(8, 9),
(1, 3),
(7, 12)(10, 15),
(2, 10)(5, 7),
(12, 15)
}
17418240

num generators = 13
[0, 1, 2, 3, 4, 5, 9, 7, 8, 6, 10, 11, 12, 13, 14]
[0, 1, 2, 3, 4, 5, 6, 8, 7, 9, 10, 11, 12, 13, 14]
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 12, 11, 10, 13, 14]
[0, 1, 2, 3, 4, 5, 6, 7, 10, 9, 8, 11, 12, 13, 14]
[0, 1, 2, 3, 4, 5, 11, 7, 8, 14, 10, 6, 12, 13, 9]
[0, 1, 2, 3, 4, 7, 6, 5, 8, 9, 10, 11, 12, 13, 14]
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 12, 14]
[0, 1, 2, 5, 4, 3, 6, 7, 8, 9, 10, 11, 12, 13, 14]
[0, 1, 3, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
[2, 1, 0, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
[0, 4, 2, 3, 1, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
[0, 9, 2, 3, 6, 5, 4, 7, 8, 1, 10, 11, 12, 13, 14]
[0, 1, 2, 3, 4, 5, 6

In [17]:
def check_codeword_preserving(perm, C_new):
    """Check if a permutation maps the codespace to itself."""
    C_set = set(C_new)
    for word in C_new:
        permuted = "".join(word[perm[i]] for i in range(len(perm)))
        if permuted not in C_set:
            return False
    return True


def check_hyperedge_preserving(perm, hyperedges):
    """Check if a permutation maps every hyperedge to another hyperedge."""
    for edge in hyperedges:
        permuted_edge = frozenset(perm[q] for q in edge)
        if permuted_edge not in hyperedges:
            return False
    return True


def filter_hypergraph_automorphisms(auts, C_new, hyperedges):
    """
    Filter automorphisms to those preserving both the codewords and the hypergraph.
    """
    valid = []
    for i, p in enumerate(auts):
        preserves_C = check_codeword_preserving(p, C_new)
        preserves_H = check_hyperedge_preserving(p, hyperedges)
        print(f"gen {i:2d}: preserves C={preserves_C}  preserves H={preserves_H}")
        if preserves_C and preserves_H:
            valid.append((i, p))
    return valid


valid_auts = filter_hypergraph_automorphisms(auts, C_new, hyperedges)
print(f"\n{len(valid_auts)} / {len(auts)} generators preserve both C and hypergraph:")
for i, p in valid_auts:
    print(f"  gen {i:2d}: {p}")

gen  0: preserves C=True  preserves H=False
gen  1: preserves C=True  preserves H=False
gen  2: preserves C=True  preserves H=False
gen  3: preserves C=True  preserves H=False
gen  4: preserves C=True  preserves H=False
gen  5: preserves C=True  preserves H=False
gen  6: preserves C=True  preserves H=False
gen  7: preserves C=True  preserves H=False
gen  8: preserves C=True  preserves H=False
gen  9: preserves C=True  preserves H=False
gen 10: preserves C=True  preserves H=False
gen 11: preserves C=True  preserves H=False
gen 12: preserves C=True  preserves H=False

0 / 13 generators preserve both C and hypergraph:


In [20]:
for name, RR_gates in [
    ("a", RR_circuit_a),
    ("b", RR_circuit_b),
    ("c", RR_circuit_c),
    ("d", RR_circuit_d),
]:
    C_piece, H_piece = apply_ccz_to_cws(C, A, RR_gates)

    valid = [
        p for p in auts
        if check_hyperedge_preserving(p, H_piece)
    ]

    print(name, "valid =", len(valid))
    for p in valid:
        print(p)

a valid = 0
b valid = 0
c valid = 0
d valid = 0


In [21]:
def three_edges_only(hyperedges):
    return {e for e in hyperedges if len(e) == 3}

def preserves_3_hyperedges(perm, hyperedges):
    H3 = three_edges_only(hyperedges)
    return {frozenset(perm[q] for q in e) for e in H3} == H3

In [22]:
valid_3edge = [
    p for p in auts
    if preserves_3_hyperedges(p, hyperedges)
]

print("valid preserving CCZ hyperedges =", len(valid_3edge))
for p in valid_3edge[:20]:
    print(p)

valid preserving CCZ hyperedges = 4
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 12, 11, 10, 13, 14]
[0, 1, 2, 3, 4, 5, 11, 7, 8, 14, 10, 6, 12, 13, 9]
[0, 1, 2, 3, 4, 7, 6, 5, 8, 9, 10, 11, 12, 13, 14]
[2, 1, 0, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]


In [23]:
def two_edges(hyperedges):
    return {e for e in hyperedges if len(e) == 2}

def edge_symmetric_difference_after_perm(perm, hyperedges):
    E = two_edges(hyperedges)
    E_perm = {frozenset(perm[q] for q in e) for e in E}
    return E_perm ^ E

for p in valid_3edge[:10]:
    diff = edge_symmetric_difference_after_perm(p, hyperedges)
    print("perm:", p)
    print("2-edge mismatch size:", len(diff))
    print(sorted([sorted(e) for e in diff]))
    print()

perm: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 12, 11, 10, 13, 14]
2-edge mismatch size: 2
[[10, 11], [11, 12]]

perm: [0, 1, 2, 3, 4, 5, 11, 7, 8, 14, 10, 6, 12, 13, 9]
2-edge mismatch size: 16
[[5, 6], [5, 9], [5, 11], [5, 14], [6, 8], [6, 10], [6, 13], [7, 9], [7, 14], [8, 11], [9, 10], [9, 12], [10, 11], [10, 14], [11, 13], [12, 14]]

perm: [0, 1, 2, 3, 4, 7, 6, 5, 8, 9, 10, 11, 12, 13, 14]
2-edge mismatch size: 2
[[5, 6], [6, 7]]

perm: [2, 1, 0, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
2-edge mismatch size: 2
[[0, 1], [1, 2]]



In [24]:
def local_complement_graph(A, v):
    """
    Apply local complementation at vertex v to adjacency matrix A (GF2).
    Toggles edges among all neighbors of v.
    Returns new adjacency matrix.
    """
    A = A.copy()
    neighbors = [u for u in range(len(A)) if A[v, u]]
    for i in range(len(neighbors)):
        for j in range(i + 1, len(neighbors)):
            u, w = neighbors[i], neighbors[j]
            A[u, w] ^= 1
            A[w, u] ^= 1
    return A


def adj_to_2edges(A):
    n = A.shape[0]
    return frozenset(
        frozenset({i, j})
        for i in range(n) for j in range(i+1, n)
        if A[i, j]
    )


def permute_adj(A, perm):
    """Apply qubit permutation to adjacency matrix."""
    n = A.shape[0]
    A_new = np.zeros((n, n), dtype=np.uint8)
    for i in range(n):
        for j in range(n):
            A_new[perm[i], perm[j]] = A[i, j]
    return A_new


def lc_equivalent_up_to_perm(A_target, A_start, max_depth=6):
    """
    BFS over sequences of local complementations to check if A_start
    can reach A_target. Returns the LC sequence if found, else None.
    """
    n = A_start.shape[0]
    target_edges = adj_to_2edges(A_target)

    # state = adjacency as bytes for hashing
    def to_key(M): return M.tobytes()

    start = A_start.copy()
    queue = [(start, [])]
    visited = {to_key(start)}

    while queue:
        cur, seq = queue.pop(0)
        if adj_to_2edges(cur) == target_edges:
            return seq
        if len(seq) >= max_depth:
            continue
        for v in range(n):
            nxt = local_complement_graph(cur, v)
            k = to_key(nxt)
            if k not in visited:
                visited.add(k)
                queue.append((nxt, seq + [v]))

    return None


# The candidate permutation
p_candidate = [2, 1, 0, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]

# Apply permutation to the graph adjacency
A_permuted = permute_adj(A, p_candidate)

print("Original 2-edges:", sorted([sorted(e) for e in adj_to_2edges(A)]))
print("Permuted 2-edges:", sorted([sorted(e) for e in adj_to_2edges(A_permuted)]))
print("Mismatch:", sorted([sorted(e) for e in adj_to_2edges(A_permuted) ^ adj_to_2edges(A)]))
print()

# Check if LC sequence on A_permuted can recover A
lc_seq = lc_equivalent_up_to_perm(A, A_permuted, max_depth=20)

if lc_seq is not None:
    print(f"✓ Found LC sequence of length {len(lc_seq)}: complement at qubits {lc_seq}")
    print("  p + LC is a valid hypergraph-CWS symmetry.")
else:
    print("✗ No LC sequence found within search depth.")
    print("  p is not a hypergraph-CWS symmetry up to LC.")

Original 2-edges: [[0, 1], [0, 3], [0, 4], [1, 3], [2, 3], [2, 4], [5, 6], [5, 8], [5, 9], [6, 8], [7, 8], [7, 9], [10, 11], [10, 13], [10, 14], [11, 13], [12, 13], [12, 14]]
Permuted 2-edges: [[0, 3], [0, 4], [1, 2], [1, 3], [2, 3], [2, 4], [5, 6], [5, 8], [5, 9], [6, 8], [7, 8], [7, 9], [10, 11], [10, 13], [10, 14], [11, 13], [12, 13], [12, 14]]
Mismatch: [[0, 1], [1, 2]]

✓ Found LC sequence of length 2: complement at qubits [3, 4]
  p + LC is a valid hypergraph-CWS symmetry.


In [25]:
# Full verification
A_permuted = permute_adj(A, p_candidate)
A_lc = A_permuted.copy()
for v in lc_seq:
    A_lc = local_complement_graph(A_lc, v)

# 1. Check 2-edges match
edges_match = adj_to_2edges(A_lc) == adj_to_2edges(A)
print(f"2-edges preserved after LC: {edges_match}")

# 2. Check 3-edges preserved under p
three_edges_orig = {e for e in hyperedges if len(e) == 3}
three_edges_perm = {frozenset(p_candidate[q] for q in e) for e in three_edges_orig}
print(f"3-edges preserved under p:  {three_edges_orig == three_edges_perm}")

# 3. Check codewords preserved under p
print(f"Codewords preserved under p: {check_codeword_preserving(p_candidate, C_new)}")

# 4. What logical operation does p induce?
print("\nLogical action of p on codewords:")
C_set = set(C_new)
for word in C_new:
    permuted = "".join(word[p_candidate[i]] for i in range(len(p_candidate)))
    print(f"  {word} -> {permuted}")

# 5. Express logical action in terms of logical Pauli basis
# Codewords index as 3-bit strings: bit0=XII, bit1=IXI, bit2=IIX
def codeword_to_logical(word, C_new):
    idx = C_new.index(word)
    # idx in binary gives the logical state
    return format(idx, f"0{3}b")

print("\nLogical mapping (XII, IXI, IIX):")
for word in C_new:
    permuted = "".join(word[p_candidate[i]] for i in range(len(p_candidate)))
    print(f"  |{codeword_to_logical(word, C_new)}> -> |{codeword_to_logical(permuted, C_new)}>")

2-edges preserved after LC: True
3-edges preserved under p:  True
Codewords preserved under p: True

Logical action of p on codewords:
  000000000000000 -> 000000000000000
  000000000001001 -> 000000000001001
  010010000000000 -> 010010000000000
  010010000001001 -> 010010000001001
  000000100100000 -> 000000100100000
  000000100101001 -> 000000100101001
  010010100100000 -> 010010100100000
  010010100101001 -> 010010100101001

Logical mapping (XII, IXI, IIX):
  |000> -> |000>
  |001> -> |001>
  |010> -> |010>
  |011> -> |011>
  |100> -> |100>
  |101> -> |101>
  |110> -> |110>
  |111> -> |111>


In [26]:
def logical_action(perm, C_new):
    C_list = list(C_new)
    C_set  = {w: i for i, w in enumerate(C_list)}
    action = []
    for word in C_list:
        permuted = "".join(word[perm[i]] for i in range(len(perm)))
        if permuted not in C_set:
            raise ValueError(f"Permutation does not preserve codespace: {word} -> {permuted}")
        action.append(C_set[permuted])
    return action


def is_identity_action(action):
    return all(action[i] == i for i in range(len(action)))


# Search all 51 candidates that preserve 3-hyperedges
print("Searching valid_3edge for nontrivial logical action + LC-graph equivalence...\n")

results = []
for p in valid_3edge:
    # Must preserve codewords
    if not check_codeword_preserving(p, C_new):
        continue

    # Check logical action
    action = logical_action(p, C_new)
    trivial = is_identity_action(action)

    # Check LC-equivalence of graph part
    A_permuted = permute_adj(A, p)
    lc_seq = lc_equivalent_up_to_perm(A, A_permuted, max_depth=6)

    print(f"perm: {p}")
    print(f"  logical action: {action}  {'(identity)' if trivial else '*** NONTRIVIAL ***'}")
    print(f"  LC sequence:    {lc_seq if lc_seq is not None else 'NOT FOUND'}")

    if lc_seq is not None and not trivial:
        results.append((p, action, lc_seq))
        print("  *** VALID NONTRIVIAL SYMMETRY FOUND ***")
    print()

print(f"\nSummary: {len(results)} nontrivial hypergraph-CWS symmetries found")
for p, action, lc_seq in results:
    print(f"  perm={p}")
    print(f"  logical action={action}")
    print(f"  LC sequence={lc_seq}")

Searching valid_3edge for nontrivial logical action + LC-graph equivalence...

perm: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 12, 11, 10, 13, 14]
  logical action: [0, 1, 2, 3, 4, 5, 6, 7]  (identity)
  LC sequence:    [13, 14]

perm: [0, 1, 2, 3, 4, 5, 11, 7, 8, 14, 10, 6, 12, 13, 9]
  logical action: [0, 4, 2, 6, 1, 5, 3, 7]  *** NONTRIVIAL ***
  LC sequence:    NOT FOUND

perm: [0, 1, 2, 3, 4, 7, 6, 5, 8, 9, 10, 11, 12, 13, 14]
  logical action: [0, 1, 2, 3, 4, 5, 6, 7]  (identity)
  LC sequence:    [8, 9]

perm: [2, 1, 0, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
  logical action: [0, 1, 2, 3, 4, 5, 6, 7]  (identity)
  LC sequence:    [3, 4]


Summary: 0 nontrivial hypergraph-CWS symmetries found


In [27]:
# Add this assertion after computing C_new to verify the assumed ordering
expected_logical_order = [
    "000000000000000",  # |000>
    "000000000001001",  # |001>
    "010010000000000",  # |010>
    "010010000001001",  # |011>
    "000000100100000",  # |100>
    "000000100101001",  # |101>
    "010010100100000",  # |110>
    "010010100101001",  # |111>
]
assert C_new == expected_logical_order, f"Codeword ordering unexpected: {C_new}"